<a href="https://colab.research.google.com/github/Raghava-Ayyagari/Quant-Trading/blob/main/Black_and_Scholes_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn


class ParametricPINN(nn.Module):
    """
    Generalized PINN: same PDE as the original model, but sigma and rho are now
    NETWORK INPUTS instead of fixed training-time constants. One trained model
    can be queried at any (t, y1, y2, sigma, rho) without retraining.

    Architecture follows the diagram: 5 -> 64 -> 128 -> 128 -> 64 -> 1
    """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, t, y1, y2, sigma, rho):
        # Each input is expected as shape (batch, 1). Concatenating along dim=1
        # (not stacking) gives the correct (batch, 5) network input.
        x = torch.cat([t, y1, y2, sigma, rho], dim=1)
        return self.net(x)

    def loss(self, t, y1, y2, r, K1, K2, sigma, rho):
        t.requires_grad_(True)
        y1.requires_grad_(True)
        y2.requires_grad_(True)

        w1 = self.forward(t, y1, y2, sigma, rho)
        w2 = self.forward(t, y2, y1, sigma, rho)

        ones = torch.ones_like(w1)

        dw1 = torch.autograd.grad(w1, y1, grad_outputs=ones, create_graph=True)[0]
        dw2 = torch.autograd.grad(w1, y2, grad_outputs=ones, create_graph=True)[0]
        dw3 = torch.autograd.grad(w1, t, grad_outputs=ones, create_graph=True)[0]

        ones_dw1 = torch.ones_like(dw1)
        ones_dw2 = torch.ones_like(dw2)

        d2wd11 = torch.autograd.grad(dw1, y1, grad_outputs=ones_dw1, create_graph=True)[0]
        d2wd22 = torch.autograd.grad(dw2, y2, grad_outputs=ones_dw2, create_graph=True)[0]
        d2wd12 = torch.autograd.grad(dw1, y2, grad_outputs=ones_dw1, create_graph=True)[0]

        val = (dw3 + rho * (sigma ** 2) * y1 * y2 * d2wd12) * (dw1 - dw2) \
            + r * (K1 * y1 + K2 * y2) * (dw1 ** 2 - dw2 ** 2) \
            - r * dw1 * (K1 * w1 + K2 * w2) \
            + r * dw2 * (K2 * w1 + K1 * w2)

        term = (K1 * (y1 ** 2) + K2 * (y2 ** 2)) * (d2wd11 * dw1 - d2wd22 * dw2) \
            - (K1 * (y2 ** 2) + K2 * (y1 ** 2)) * (d2wd11 * dw2 - d2wd22 * dw1)
        val = val + 0.5 * sigma * sigma * term

        return torch.mean(val ** 2)

    def basecase_loss(self, y1, y2, t0, c, k1, k2, sigma, rho):
        t = torch.full_like(y1, t0)
        w1 = self.forward(t, y1, y2, sigma, rho)
        return torch.mean((w1 - torch.relu(k1 * y1 + k2 * y2 - c)) ** 2)

    def set(self, t, y1, y2, sigma, rho):
        def col(x):
            return torch.as_tensor(x, dtype=torch.float32).reshape(-1, 1)

        t = col(t)
        y1 = col(y1).requires_grad_(True)
        y2 = col(y2).requires_grad_(True)
        sigma = col(sigma)
        rho = col(rho)

        w1 = self.forward(t, y1, y2, sigma, rho)
        w2 = self.forward(t, y2, y1, sigma, rho)

        ones = torch.ones_like(w1)
        dw1 = torch.autograd.grad(w1, y1, grad_outputs=ones, create_graph=True, allow_unused=True)[0]
        dw2 = torch.autograd.grad(w1, y2, grad_outputs=ones, create_graph=True, allow_unused=True)[0]

        scalar_factor = 1.0 / (dw1 ** 2 - dw2 ** 2)
        row1 = torch.stack([dw1, -dw2])
        row2 = torch.stack([-dw2, dw1])
        matrix = torch.stack([row1, row2])
        A = scalar_factor * matrix

        return w1, w2, A


def train(
    model,
    epochs=20000,
    batch_size=1000,
    y_range=(50.0, 250.0),
    t_range=(0.0, 2.0),
    sigma_range=(0.05, 0.5),
    rho_range=(-0.9, 0.9),
    r=0.05,
    K1=0.5,
    K2=0.5,
    K=100.0,
    T=2.0,
    lr=1e-3,
    bc_weight=0.1,
    device="cpu",
):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

    history = []

    for epoch in range(epochs):
        optimizer.zero_grad()

        t = (torch.rand(batch_size, 1, device=device) * (t_range[1] - t_range[0]) + t_range[0])
        y1 = (torch.rand(batch_size, 1, device=device) * (y_range[1] - y_range[0]) + y_range[0])
        y2 = (torch.rand(batch_size, 1, device=device) * (y_range[1] - y_range[0]) + y_range[0])
        sigma = (torch.rand(batch_size, 1, device=device) * (sigma_range[1] - sigma_range[0]) + sigma_range[0])
        rho = (torch.rand(batch_size, 1, device=device) * (rho_range[1] - rho_range[0]) + rho_range[0])

        pde_loss = model.loss(t, y1, y2, r, K1, K2, sigma, rho)

        y1_bc = (torch.rand(batch_size, 1, device=device) * (y_range[1] - y_range[0]) + y_range[0])
        y2_bc = (torch.rand(batch_size, 1, device=device) * (y_range[1] - y_range[0]) + y_range[0])
        sigma_bc = (torch.rand(batch_size, 1, device=device) * (sigma_range[1] - sigma_range[0]) + sigma_range[0])
        rho_bc = (torch.rand(batch_size, 1, device=device) * (rho_range[1] - rho_range[0]) + rho_range[0])

        bc_loss = model.basecase_loss(y1_bc, y2_bc, T, K, K1, K2, sigma_bc, rho_bc)

        loss = pde_loss + bc_weight * bc_loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        history.append((pde_loss.item(), bc_loss.item()))

        if epoch % 1000 == 0:
            print(f"epoch {epoch:5d}  pde_loss={pde_loss.item():.6f}  bc_loss={bc_loss.item():.6f}")

    return history


# --- Initialize Model and Push to GPU ---
pricing_function = ParametricPINN().to("cuda")
pricing_function.load_state_dict(torch.load('/content/drive/MyDrive/Black_and_Scholes_Model/model.pth', weights_only=True))
# --- Train and Save ---


<All keys matched successfully>

In [ ]:
if __name__ == "__main__":
  torch.manual_seed(0)
  hist = train(pricing_function, epochs=10000)
  torch.save(pricing_function.state_dict(), "parametric_model.pth")
  torch.save(pricing_function.state_dict(), '/content/drive/MyDrive/Black_and_Scholes_Model/model.pth')

epoch     0  pde_loss=0.143995  bc_loss=0.077233
epoch  1000  pde_loss=0.059662  bc_loss=1.403256
epoch  2000  pde_loss=0.079259  bc_loss=0.452041
epoch  3000  pde_loss=0.164591  bc_loss=0.637501
epoch  4000  pde_loss=0.052171  bc_loss=0.286983
epoch  5000  pde_loss=0.044374  bc_loss=0.278862
epoch  6000  pde_loss=0.041747  bc_loss=0.220338
epoch  7000  pde_loss=0.035458  bc_loss=0.183589
epoch  8000  pde_loss=0.043984  bc_loss=0.201097
epoch  9000  pde_loss=0.018344  bc_loss=0.191695
